# Wine Sales Forecasting with Rolling-Origin Validation

**Recruiter-facing end-to-end analysis · Monthly time-series forecasting · Python 3.12/3.13**

> Rolling-origin selection chooses trend plus month for Rose (holdout RMSE 13.6) and additive Holt-Winters for Sparkling (RMSE 358.7).

## Executive summary

**Objective:** Select transparent monthly forecasting methods on multiple historical origins and evaluate once on a final 24-month holdout.

**Data:** 187 monthly Rose and Sparkling observations from January 1980 through July 1995.

**Verified result:** Rolling-origin selection chooses trend plus month for Rose (holdout RMSE 13.6) and additive Holt-Winters for Sparkling (RMSE 358.7).

**Decision supported:** Choose a reproducible baseline and quantify forecast risk before inventory commitment.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** An inventory or demand-planning analyst.

**Decision:** Choose a reproducible baseline and quantify forecast risk before inventory commitment.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '09-wine-sales-forecasting'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.10
pandas            2.3.3
NumPy             2.5.2
SciPy            1.18.0
scikit-learn      1.9.0
Matplotlib       3.11.1

Project: 09-wine-sales-forecasting


## 4. Data provenance and scope

Bundled in the original repository; commercial source, currency/units, and redistribution terms are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

         file  size_mb           sha256
     rose.csv    0.002 e3553bd7f11fa3c8
sparkling.csv    0.003 539745fc03e143ea


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


rose.csv: 2 columns
YearMonth  Rose
  1980-01   112
  1980-02   118
  1980-03   129
  1980-04    99
  1980-05   116

sparkling.csv: 2 columns
YearMonth  Sparkling
  1980-01       1686
  1980-02       1591
  1980-03       2304
  1980-04       1712
  1980-05       1471


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 155 lines
Functions: _seasonal_naive, _trend_month, _ridge_lags, _holt_winters_additive, _forecast, _ljung_box, _evaluate_series, run_analysis


## 7. Methodology and hypotheses

Frequency audit, train-contained interpolation, seasonal-naive baseline, trend/month regression, lagged ridge, additive Holt-Winters implementation, three rolling origins, empirical intervals, and Ljung–Box residual autocorrelation.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_09_wine_sales_forecasting", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 0.42 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


rose_data_quality.csv (2 fields)
   column          dtype  missing_count  missing_percent  unique_values  constant
YearMonth datetime64[ns]              0             0.00            187     False
     Rose        float64              2             1.07             97     False

sparkling_data_quality.csv (2 fields)
   column          dtype  missing_count  missing_percent  unique_values  constant
YearMonth datetime64[ns]              0              0.0            187     False
Sparkling          int64              0              0.0            176     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'rolling_origin_results.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Rolling-origin selection chooses trend plus month for Rose (holdout RMSE 13.6) and additive Holt-Winters for Sparkling (RMSE 358.7).')

Primary evidence: rolling_origin_results.csv, shape=(36, 7)
product  fold                      model     mae    rmse  r_squared  mape_percent
   rose     1             seasonal_naive  9.8333 11.4673     0.7540       14.5535
   rose     1           trend_plus_month 11.8469 14.0723     0.6295       16.2134
   rose     1             ridge_lag_1_12 13.6762 14.4725     0.6081       19.6624
   rose     1 holt_winters__0.2_0.05_0.2 10.9914 12.8388     0.6916       14.2586
   rose     1  holt_winters__0.3_0.1_0.3 11.0270 12.6525     0.7005       15.1359
   rose     1  holt_winters__0.5_0.1_0.2 17.1773 20.1825     0.2379       23.6140
   rose     2             seasonal_naive 15.5833 18.3553    -0.1606       25.9618
   rose     2           trend_plus_month  6.6404 10.0496     0.6521       10.0180
   rose     2             ridge_lag_1_12 18.0409 19.5282    -0.3136       34.2459
   rose     2 holt_winters__0.2_0.05_0.2 17.5277 19.1709    -0.2660       29.0934
   rose     2  holt_winters__0.3_0.1_0

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


VALIDATION
{
  "model_selection": "three rolling-origin 12-month validation windows",
  "final_holdout": "last 24 months, untouched during selection",
  "missing_rose_values": "linear interpolation within the series; neither missing record is in the final holdout"
}


## 12. Visual evidence

### Chronological Holdout Forecasts

![chronological_holdout_forecasts](../reports/figures/chronological_holdout_forecasts.png)

### Wine Forecasting Evidence

![wine_forecasting_evidence](../reports/figures/wine_forecasting_evidence.png)

## 13. Business interpretation

Rolling-origin selection chooses trend plus month for Rose (holdout RMSE 13.6) and additive Holt-Winters for Sparkling (RMSE 358.7).

The correct action is to use this result as evidence for **Choose a reproducible baseline and quantify forecast risk before inventory commitment.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

The series ends in 1995 and lacks price, promotion, inventory, weather, and economic drivers; diagnostic intervals are empirical validation-error bands rather than formal probabilistic intervals.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                           artifact  size_kb       sha256
reports\figures\chronological_holdout_forecasts.png    215.7 b8223b07f47c
      reports\figures\wine_forecasting_evidence.png    320.2 d5bbf884df08
                               reports\metrics.json      4.9 889cd496fc05
          reports\tables\rolling_origin_results.csv      3.9 d01297c8942c
               reports\tables\rose_data_quality.csv      0.1 df1f7f8669cf
              reports\tables\rose_final_holdout.csv      1.8 664136a3a837
           reports\tables\rose_model_comparison.csv      0.6 2ad0bb391ab4
          reports\tables\sparkling_data_quality.csv      0.1 55d0a9b15880
         reports\tables\sparkling_final_holdout.csv      1.8 f1a64e11fe0f
      reports\tables\sparkling_model_comparison.csv      0.6 7087865ac676


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed select transparent monthly forecasting methods on multiple historical origins and evaluate once on a final 24-month holdout. using frequency audit, train-contained interpolation, seasonal-naive baseline, trend/month regression, lagged ridge, additive holt-winters implementation, three rolling origins, empirical intervals, and ljung–box residual autocorrelation. The final verified conclusion is: **Rolling-origin selection chooses trend plus month for Rose (holdout RMSE 13.6) and additive Holt-Winters for Sparkling (RMSE 358.7).** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/09-wine-sales-forecasting/src/analysis.py
python scripts/execute_notebooks.py --project 09-wine-sales-forecasting
```